# Figures

Reads the result tables written by Phases 9 to 13 and produces the manuscript
figures.

**Inputs:** `data/results/phase09_results.csv`,
`data/results/phase10_heterogeneity.csv`,
`data/results/phase12_results.csv`, `data/results/phase12_tost.csv`,
`data/results/phase13_callaway_santanna.csv`,
`data/results/phase13_cohort_atts.csv`,
`data/interim/phase11_panel_controls.csv`,
`data/interim/phase06_matches.csv`

**Outputs:** PNG at 300 dpi in `figures/png/`, PDF in `figures/pdf/`

| Figure | Supports |
|---|---|
| 2 | Covariate balance after matching |
| 3 | Observed publishing rates, no model |
| 4 | Publishing activity against matched controls, by arm |
| 5 | Publishing-activity effects by arm, with equivalence bounds |
| 6 | Lead against middle authors, with placebo |
| 7 | Lead and middle authors, observed output |
| 8 | Annual output, both margins |
| 9 | Subgroup effects against chance |
| 10 | The cohort gradient |


In [1]:
import os

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

R09 = "data/results/phase09_results.csv"
R10 = "data/results/phase10_heterogeneity.csv"
R12 = "data/results/phase12_results.csv"
TOST = "data/results/phase12_tost.csv"
R13 = "data/results/phase13_callaway_santanna.csv"
R13_COHORT = "data/results/phase13_cohort_atts.csv"
PANEL = "data/interim/phase11_panel_controls.csv"
MATCHES = "data/interim/phase06_matches.csv"

OUTDIR_PNG = "figures/png"
OUTDIR_PDF = "figures/pdf"

ARMS = ["AUTHOR_MISCONDUCT", "HONEST_ERROR", "EDITORIAL_COMPROMISE"]
REF_ARM = "CONTROL"
REF_EVENTS = (-2, -1)
ANALYSIS_PRE, ANALYSIS_POST = 6, 6
HORIZON = 6
BOUND_EXIT = 0.02

LABEL = {"AUTHOR_MISCONDUCT": "Author misconduct",
         "HONEST_ERROR": "Honest error",
         "EDITORIAL_COMPROMISE": "Editorial compromise",
         "CONTROL": "Matched controls",
         "True": "Lead authors (first or last)",
         "False": "Middle authors"}
COLOUR = {"AUTHOR_MISCONDUCT": "#B23A48", "HONEST_ERROR": "#3E6680",
          "EDITORIAL_COMPROMISE": "#7A8450", "CONTROL": "#6E6E6E",
          "True": "#B23A48", "False": "#3E6680"}
MARKER = {"AUTHOR_MISCONDUCT": "o", "HONEST_ERROR": "s",
          "EDITORIAL_COMPROMISE": "^", "CONTROL": "D",
          "True": "o", "False": "s"}

plt.rcParams.update({
    "figure.dpi": 140, "savefig.dpi": 300,
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "legend.fontsize": 8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "figure.facecolor": "white",
})
os.makedirs(OUTDIR_PNG, exist_ok=True)
os.makedirs(OUTDIR_PDF, exist_ok=True)

print(f"writing to {OUTDIR_PNG}/ and {OUTDIR_PDF}/")
for label, path in [("Phase 9", R09), ("Phase 10", R10), ("Phase 12", R12),
                    ("TOST", TOST), ("Phase 13", R13),
                    ("cohort ATTs", R13_COHORT), ("panel", PANEL),
                    ("matches", MATCHES)]:
    print(f"  {label:<14}{'found' if os.path.isfile(path) else 'MISSING'}")

writing to figures/png/ and figures/pdf/
  Phase 9       found
  Phase 10      found
  Phase 12      found
  TOST          found
  Phase 13      found
  cohort ATTs   found
  panel         found
  matches       found


In [2]:
def save(fig, name):
    fig.savefig(f"{OUTDIR_PNG}/{name}.png", bbox_inches="tight")
    fig.savefig(f"{OUTDIR_PDF}/{name}.pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"  wrote {name}")


def load(path):
    return pd.read_csv(path, low_memory=False) if os.path.isfile(path) else None


def mark_reference(ax, label=True):
    """Shade the omitted event times.

    Coefficients are differences from that period, not from an absolute
    baseline, so zero on the vertical axis is not a level.
    """
    ax.axvspan(REF_EVENTS[0] - 0.5, REF_EVENTS[-1] + 0.5,
               color="#000000", alpha=0.055, lw=0, zorder=0)
    ax.axvline(-0.5, color="#333333", lw=0.9, ls=(0, (4, 3)), zorder=1)
    ax.axhline(0, color="#333333", lw=0.7, zorder=1)
    if label:
        ax.text(np.mean(REF_EVENTS), ax.get_ylim()[1], "reference\nperiod",
                ha="center", va="top", fontsize=7, color="#555555")


def gapped(df, xcol="event_time", ycol="coef", secol="se"):
    """Insert the omitted event times as NaN so the line breaks there.

    Drawn straight across, the line would show a trajectory through two points
    that were never estimated.
    """
    d = df[[xcol, ycol, secol]].copy()
    gaps = pd.DataFrame({xcol: list(REF_EVENTS), ycol: np.nan, secol: np.nan})
    return (pd.concat([d, gaps], ignore_index=True)
              .sort_values(xcol).reset_index(drop=True))


def band(ax, x, y, se, colour, alpha=0.13):
    ok = ~np.isnan(y)
    for seg in np.split(np.flatnonzero(ok),
                        np.flatnonzero(np.diff(np.flatnonzero(ok)) > 1) + 1):
        if len(seg):
            ax.fill_between(x[seg], y[seg] - 1.96 * se[seg],
                            y[seg] + 1.96 * se[seg],
                            color=colour, alpha=alpha, lw=0)

## Figure 2 — Covariate balance

Standardised differences between treated authors and their matched controls.
The design constrains publication volume in two pre-retraction windows and the
ratio between them, so that the groups are comparable in level and in
direction of output before the event.

In [3]:
m = load(MATCHES)
if m is None:
    print("matches file missing; figure 2 skipped")
else:
    pairs = [("treated_pre_pubs", "control_pre_pubs",
              "Publications, 7 years before"),
             ("treated_pre_recent", "control_pre_recent",
              "Publications, most recent 3 years"),
             ("treated_pre_early", "control_pre_early",
              "Publications, earlier 4 years"),
             ("treated_pre_growth", "control_pre_growth",
              "Pre-period growth (log rate ratio)")]
    rows = []
    for t, c, label in pairs:
        if t not in m.columns or c not in m.columns:
            continue
        a, b = m[t], m[c]
        pooled = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
        rows.append((label, (b.mean() - a.mean()) / pooled if pooled else 0.0))

    if not rows:
        print("no balance columns; figure 2 skipped")
    else:
        fig, ax = plt.subplots(figsize=(6.4, 2.6))
        y = np.arange(len(rows))[::-1]
        ax.axvspan(-0.1, 0.1, color="#4477AA", alpha=0.07, lw=0)
        ax.axvline(0, color="#333333", lw=0.8)
        ax.scatter([v for _, v in rows], y, s=44, color="#B23A48", zorder=3)
        ax.set_yticks(y)
        ax.set_yticklabels([lab for lab, _ in rows])
        ax.set_xlim(-0.2, 0.2)
        ax.grid(axis="y", visible=False)
        ax.set_xlabel("Standardised difference, control minus treated")
        ax.text(0.1, -0.55, "conventional threshold", fontsize=6.5,
                color="#4477AA", ha="center", va="top")
        ax.set_ylim(-0.9, len(rows) - 0.5)
        save(fig, "fig2_covariate_balance")

  wrote fig2_covariate_balance


## Figure 3 — Observed publishing rates

No model. The share of each group publishing in each year relative to the
event.

In [4]:
if not os.path.isfile(PANEL):
    print("panel missing; figure 3 skipped")
else:
    p = pd.read_csv(PANEL, usecols=["event_time", "arm", "active"],
                    low_memory=False)
    p = p[(p.event_time >= -ANALYSIS_PRE) & (p.event_time <= ANALYSIS_POST)]

    fig, ax = plt.subplots(figsize=(6.6, 4.2))
    for arm in ARMS + [REF_ARM]:
        d = p[p.arm == arm]
        if d.empty:
            continue
        s = d.groupby("event_time").active.mean()
        ax.plot(s.index, s.values, color=COLOUR[arm],
                lw=1.9 if arm == REF_ARM else 1.5,
                ls="--" if arm == REF_ARM else "-",
                marker=MARKER[arm], ms=3.6, label=LABEL[arm],
                zorder=4 if arm == REF_ARM else 3)

    ax.set_xlabel("Years relative to retraction "
                  "(0 is the year the notice appeared)")
    ax.set_ylabel("Share of authors publishing that year")
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.set_xlim(-ANALYSIS_PRE - 0.4, ANALYSIS_POST + 0.4)
    ax.axvline(-0.5, color="#333333", lw=0.9, ls=(0, (4, 3)))
    ax.legend(frameon=False, loc="lower left",
              bbox_to_anchor=(0, 1.02), ncol=2)
    save(fig, "fig3_observed_rates")

  wrote fig3_observed_rates


## Figure 4 — Publishing activity against matched controls

The main event study. Pre-period coefficients are shown rather than omitted:
they are not flat, and the figure is the place a reader should be able to see
that.

In [5]:
r12 = load(R12)
if r12 is None:
    print("Phase 12 results missing; figure 4 skipped")
else:
    d12 = r12[r12.outcome == "active"]
    fig, ax = plt.subplots(figsize=(6.6, 4.2))

    for arm in ARMS:
        s = d12[d12.group == arm]
        if s.empty:
            continue
        g = gapped(s)
        x, y, se = g.event_time.values, g.coef.values, g.se.values
        band(ax, x, y, se, COLOUR[arm])
        ax.plot(x, y, color=COLOUR[arm], lw=1.6, marker=MARKER[arm], ms=4,
                label=LABEL[arm], zorder=3)

    ax.plot([], [], color=COLOUR[REF_ARM], lw=1.4, ls="--",
            label="Matched controls (zero by construction)")

    ax.set_xlabel("Years relative to retraction "
                  "(0 is the year the notice appeared)")
    ax.set_ylabel("Difference from matched controls\n"
                  "in probability of publishing")
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.set_xlim(-ANALYSIS_PRE - 0.4, ANALYSIS_POST + 0.4)
    mark_reference(ax)
    ax.legend(frameon=False, loc="lower left",
              bbox_to_anchor=(0, 1.02), ncol=2)
    save(fig, "fig4_exit_event_study")

  wrote fig4_exit_event_study


## Figure 5 — Publishing-activity effects by arm, with equivalence bounds

Both estimators at the reporting horizon, alongside the equivalence tests.
Panel A shows the effect by arm under the conventional and staggered-adoption
specifications. Panel B shows the three arm contrasts against the bound
declared in advance and the tightest bound the data achieve. If the equivalence
table is absent, Panel A is drawn alone.


In [6]:
r12 = load(R12)
r13 = load(R13)
tt = load(TOST)
if r12 is None:
    print("Phase 12 results missing; figure 5 skipped")
else:
    d12 = r12[(r12.outcome == "active") & (r12.event_time == HORIZON)]
    rows = []
    for arm in ARMS:
        a = d12[d12.group == arm]
        if a.empty:
            continue
        cs = None
        if r13 is not None:
            c = r13[(r13.arm == arm) & (r13.event_time == HORIZON)]
            if len(c):
                cs = (float(c.att.iloc[0]), float(c.se.iloc[0]))
        rows.append((arm, (float(a.coef.iloc[0]), float(a.se.iloc[0])), cs))

    # Panel B data from the TOST table, tolerant of column naming.
    contrasts = None
    sel = None
    if tt is not None:
        sel = tt[(tt.outcome == "active") & (tt.k == HORIZON)].copy()
        if len(sel):
            def pick(cols):
                return next((c for c in cols if c in sel.columns), None)
            ccol = pick(["contrast", "label", "pair", "comparison"])
            dcol = pick(["diff", "difference", "estimate", "coef"])
            scol = pick(["se", "std_err", "stderr"])
            locol = pick(["lo90", "ci_lo", "ci_lower", "lower", "lo"])
            hicol = pick(["hi90", "ci_hi", "ci_upper", "upper", "hi"])
            mid = lo = hi = None
            if locol and hicol:
                lo = sel[locol].astype(float).values
                hi = sel[hicol].astype(float).values
                mid = (sel[dcol].astype(float).values if dcol
                       else (lo + hi) / 2)
            elif dcol and scol:
                mid = sel[dcol].astype(float).values
                half = 1.645 * sel[scol].astype(float).values
                lo, hi = mid - half, mid + half
            if mid is not None:
                names = (sel[ccol].astype(str).tolist() if ccol
                         else [f"contrast {i + 1}" for i in range(len(mid))])
                mapping = [("AUTHOR_MISCONDUCT", "AM"),
                           ("HONEST_ERROR", "HE"),
                           ("EDITORIAL_COMPROMISE", "EC"),
                           ("AUTH", "AM"),
                           ("HONE", "HE"),
                           ("EDIT", "EC")]

                def relabel(n):
                    for k, v in mapping:
                        n = n.replace(k, v)
                    return (n.replace("_vs_", " \u2212 ")
                             .replace(" - ", " \u2212 ")
                             .replace("-", " \u2212 "))

                names = [relabel(n) for n in names]
                declared = (float(sel.bound.iloc[0])
                            if "bound" in sel.columns else BOUND_EXIT)
                achieved = (float(sel.min_bound.max())
                            if "min_bound" in sel.columns
                            else float(np.abs(np.c_[lo, hi]).max()))
                contrasts = (names, mid, lo, hi, declared, achieved)

    ncols = 2 if contrasts else 1
    fig, axes = plt.subplots(1, ncols, figsize=(5.1 * ncols, 3.6),
                             squeeze=False)
    if ncols == 2:
        fig.subplots_adjust(wspace=0.32)
    axA = axes[0][0]
    ypos = np.arange(len(rows))[::-1].astype(float)
    OFF = 0.16

    for y, (arm, tw, cs) in zip(ypos, rows):
        axA.errorbar(tw[0], y + OFF, xerr=1.96 * tw[1], fmt=MARKER[arm],
                     color=COLOUR[arm], ms=6, capsize=3, lw=1.5, zorder=3)
        if cs is not None:
            axA.errorbar(cs[0], y - OFF, xerr=1.96 * cs[1], fmt=MARKER[arm],
                         color=COLOUR[arm], ms=6, capsize=3, lw=1.5,
                         mfc="white", zorder=3)

    axA.axvline(0, color="#333333", lw=0.8)
    axA.set_yticks(ypos)
    axA.set_yticklabels([LABEL[a] for a, _, _ in rows])
    axA.set_ylim(-0.75, len(rows) - 0.25)
    axA.grid(axis="y", visible=False)

    vals = [cs[0] for _, _, cs in rows if cs] or [tw[0] for _, tw, _ in rows]
    if len(vals) > 1:
        lo_v, hi_v = min(vals), max(vals)
        axA.plot([lo_v, hi_v], [-0.35, -0.35], color="#666666", lw=1.2,
                 marker="|", ms=7)
        axA.text((lo_v + hi_v) / 2, -0.43,
                 f"spread across arms: {100 * abs(hi_v - lo_v):.2f} points",
                 ha="center", va="top", fontsize=7, color="#666666")

    axA.set_xlabel(f"Change in the probability of publishing at +{HORIZON},\n"
                   f"against matched never-retracted colleagues", labelpad=6)
    if contrasts:
        axA.set_title("A. Effect by arm, both estimators", loc="left",
                      fontsize=9, pad=8)
    axA.plot([], [], "ko", ms=5, label="Two-way fixed effects")
    if any(cs for _, _, cs in rows):
        axA.plot([], [], "ko", ms=5, mfc="white", label="Callaway-Sant'Anna")
    axA.legend(frameon=False, loc="upper left", fontsize=7.5)
    if contrasts:
        names, mid, lo, hi, declared, achieved = contrasts
        axB = axes[0][1]
        yb = np.arange(len(mid))[::-1].astype(float)
        axB.axvspan(-declared, declared, color="#4477AA", alpha=0.10, lw=0)
        for v in (-achieved, achieved):
            axB.axvline(v, color="#666666", lw=0.9, ls=(0, (4, 3)))
        axB.axvline(0, color="#333333", lw=0.8)
        axB.errorbar(mid, yb, xerr=np.vstack([mid - lo, hi - mid]), fmt="o",
                     color="#333333", ms=5, capsize=3, lw=1.4, zorder=3)
        axB.set_yticks(yb)
        axB.set_yticklabels(names, fontsize=8.5)
        axB.grid(axis="y", visible=False)
        span = 1.3 * max(achieved, float(np.abs(lo).max()),
                         float(np.abs(hi).max()))
        axB.set_xlim(-span, span)
        axB.set_ylim(-0.9, len(mid) - 0.25)
        axB.text(0, len(mid) - 0.55, f"declared bound \u00b1{declared:g}",
                 ha="center", va="bottom", fontsize=6.5, color="#4477AA")
        axB.text(achieved, -0.45, f"achieved \u00b1{achieved:.3f} ",
                 ha="right", va="top", fontsize=6.5, color="#666666")
        axB.set_title("B. Arm contrasts against the equivalence bounds",
                      loc="left", fontsize=9, pad=8)
        axB.set_xlabel(f"Contrast at +{HORIZON} (90% interval)", labelpad=6)
        axB.text(0.5, -0.30,
                 "AM author misconduct   HE honest error   "
                 "EC editorial compromise",
                 transform=axB.transAxes, ha="center", va="top",
                 fontsize=6.8, color="#555555")
    save(fig, "fig5_invariance")


  wrote fig5_invariance


## Figure 6 — Lead against middle authors

Both groups are named on the same retraction notice, so field, venue, era and
the notice itself are held constant and only attributed responsibility differs.
The placebo places a fake event six years earlier, outside the pre-retraction
rise.

In [7]:
r09 = load(R09)
if r09 is None:
    print("Phase 9 results missing; figure 6 skipped")
else:
    real = r09[r09.spec == "publications ~ position"]
    plac = r09[r09.spec == "PLACEBO publications ~ position"]

    if real.empty:
        print("no position estimates; figure 6 skipped")
    else:
        srcs = [(real, "At the real retraction")]
        if not plac.empty:
            srcs.append((plac, "At a fake event, six years earlier"))

        xs = [v for s, _ in srcs for v in s.event_time.values]
        lo, hi = min(xs), max(xs)

        fig, axes = plt.subplots(1, len(srcs), figsize=(4.4 * len(srcs), 3.8),
                                 sharey=True, sharex=True, squeeze=False)
        axes = axes[0]
        for ax, (src, ttl) in zip(axes, srcs):
            for g in ["True", "False"]:
                s = src[src.group.astype(str) == g]
                if s.empty:
                    continue
                gg = gapped(s)
                x, y, se = gg.event_time.values, gg.coef.values, gg.se.values
                band(ax, x, y, se, COLOUR[g], 0.12)
                ax.plot(x, y, color=COLOUR[g], lw=1.6, marker=MARKER[g], ms=4,
                        label=LABEL[g])
            ax.set_title(ttl, loc="left", fontsize=9, pad=8)
            ax.set_xlabel("Years relative to the event")
            ax.xaxis.set_major_locator(MultipleLocator(2))
            ax.set_xlim(lo - 0.4, hi + 0.4)
            mark_reference(ax, label=False)

        axes[0].set_ylabel("Change in annual publications")
        h, l = axes[0].get_legend_handles_labels()
        fig.legend(h, l, frameon=False, loc="lower center",
                   bbox_to_anchor=(0.5, 0.99), ncol=2)
        save(fig, "fig6_position_and_placebo")

  wrote fig6_position_and_placebo


## Figure 7 — Lead against middle authors, observed output

The same two groups as Figure 6, as raw annual means rather than regression
coefficients, and windowed on the real retraction and on a placebo date.

The two figures answer different questions. Figure 6 reports estimates measured
relative to the omitted reference years, after author fixed effects have removed
each author's own average level; zero there means "unchanged from this author's
own baseline". Here zero means zero papers, no baseline is subtracted, and
differences in career stage and publishing volume between the two groups remain
in the series.

The right-hand panels centre the same series on a date four years before the
retraction, a window lying entirely within the pre-period. Nothing happens
there, so the pipeline peak at the event year and the separation after it
should both be absent.

The estimation in Phase 9 places its placebo at six years. The panel spans −6 to
+6, so a window centred there has no years before it and would reach past the
real retraction; four is the closest date that supports a window on both sides.
The two columns are drawn from one series and share data by construction: the
placebo is a different window on it rather than a separate estimation.

In [8]:
PANEL_TREATED = "data/interim/phase08_panel_extended.csv"

# The estimation in Phase 9 places its placebo six years before the retraction.
# The panel spans -6 to +6, so a window centred there has no years before it,
# and a window wide enough to show a post-event response would reach past the
# real event. Centring at -4 gives a window lying entirely within the
# pre-period while keeping the post-event span in which a separation would
# appear.
FIG_PLACEBO_OFFSET = -4
WIN_PRE, WIN_POST = 2, 3

if not os.path.isfile(PANEL_TREATED):
    print("treated panel missing; figure 7 skipped")
else:
    p = pd.read_csv(PANEL_TREATED, low_memory=False)
    p = p[p.balanced & p.in_primary]
    p = p[p.first_position.astype(str).isin(["first", "middle", "last"])].copy()
    p["is_lead"] = p.first_position.astype(str).isin(["first", "last"])

    outcomes = [(o, lab) for o, lab in
                [("publications", "Papers per year"),
                 ("active", "Share publishing that year")]
                if o in p.columns]

    fig, axes = plt.subplots(len(outcomes), 2,
                             figsize=(8.8, 3.5 * len(outcomes)),
                             squeeze=False)

    for row, (outcome, ylabel) in enumerate(outcomes):
        for col, (offset, ttl) in enumerate(
                [(0, "At the real retraction"),
                 (FIG_PLACEBO_OFFSET,
                  f"At a placebo date, {abs(FIG_PLACEBO_OFFSET)} years "
                  f"earlier")]):
            ax = axes[row][col]
            d = p.copy()
            d["t"] = d.event_time - offset
            d = d[(d.t >= -WIN_PRE) & (d.t <= WIN_POST)]

            for lead, key in [(True, "True"), (False, "False")]:
                s = d[d.is_lead == lead].groupby("t")[outcome]
                mean, se = s.mean(), s.sem()
                ax.fill_between(mean.index, mean - 1.96 * se, mean + 1.96 * se,
                                color=COLOUR[key], alpha=0.13, lw=0)
                ax.plot(mean.index, mean.values, color=COLOUR[key], lw=1.6,
                        marker=MARKER[key], ms=4.4, label=LABEL[key])

            ax.axvline(-0.5, color="#333333" if col == 0 else "#888888",
                       lw=1.0, ls=(0, (4, 3)) if col == 0 else (0, (2, 3)))
            if row == 0:
                ax.set_title(ttl, loc="left", fontsize=9, pad=8)
            if row == len(outcomes) - 1:
                ax.set_xlabel("Years relative to the event")
            if col == 0:
                ax.set_ylabel(ylabel)
            ax.xaxis.set_major_locator(MultipleLocator(1))
            ax.set_xlim(-WIN_PRE - 0.3, WIN_POST + 0.3)

        # Shared vertical scale within a row, so the columns are comparable.
        y0 = min(axes[row][c].get_ylim()[0] for c in (0, 1))
        y1 = max(axes[row][c].get_ylim()[1] for c in (0, 1))
        for c in (0, 1):
            axes[row][c].set_ylim(y0, y1)

    h, l = axes[0][0].get_legend_handles_labels()
    fig.legend(h, l, frameon=False, loc="lower center",
               bbox_to_anchor=(0.5, 1.0), ncol=2)
    save(fig, "fig7_position_observed")

  wrote fig7_position_observed


## Figure 8 — Annual output, both margins

The manuscript reports this outcome as uninformative. The figure shows the
converging pre-period that disqualifies it.


In [9]:
r12 = load(R12)
if r12 is None:
    print("Phase 12 results missing; figure 8 skipped")
else:
    have = set(r12.outcome.unique())
    panels = [(o, t) for o, t in
              [("publications",
                "All authors\n(a zero counted for years not publishing)"),
               ("publications_active_only",
                "Authors still publishing\n(intensive margin)")]
              if o in have]

    if not panels:
        print("no publication outcomes; figure 8 skipped")
    else:
        fig, axes = plt.subplots(1, len(panels),
                                 figsize=(4.4 * len(panels), 3.7), sharey=True)
        axes = np.atleast_1d(axes)
        for ax, (outcome, title) in zip(axes, panels):
            for arm in ARMS:
                s = r12[(r12.outcome == outcome) & (r12.group == arm)]
                if s.empty:
                    continue
                g = gapped(s)
                x, y, se = g.event_time.values, g.coef.values, g.se.values
                band(ax, x, y, se, COLOUR[arm], 0.11)
                ax.plot(x, y, color=COLOUR[arm], lw=1.5, marker=MARKER[arm],
                        ms=3.6, label=LABEL[arm])
            ax.set_title(title, loc="left", fontsize=8.5, pad=8)
            ax.set_xlabel("Years relative to retraction")
            ax.xaxis.set_major_locator(MultipleLocator(2))
            ax.set_xlim(-ANALYSIS_PRE - 0.4, ANALYSIS_POST + 0.4)
            mark_reference(ax, label=False)

        axes[0].set_ylabel("Difference from matched controls\n"
                           "(papers per year)")
        h, l = axes[0].get_legend_handles_labels()
        fig.legend(h, l, frameon=False, loc="lower center",
                   bbox_to_anchor=(0.5, 1.02), ncol=3)
        save(fig, "fig8_output_both_margins")

  wrote fig8_output_both_margins


## Figure 9 — Subgroup effects against chance

Every pre-specified subgroup test, ordered by p-value, against the uniform
distribution expected if no subgroup differs.

In [10]:
r10 = load(R10)
if r10 is None:
    print("Phase 10 results missing; figure 9 skipped")
else:
    het = r10[r10.table == "heterogeneity_test"] if "table" in r10.columns \
        else r10
    if "p" not in het.columns or het.empty:
        print("no heterogeneity tests; figure 9 skipped")
    else:
        p = np.sort(het.p.dropna().values)
        n = len(p)
        expected = (np.arange(1, n + 1) - 0.5) / n

        fig, ax = plt.subplots(figsize=(5.0, 4.4))
        ax.plot([0, 1], [0, 1], color="#999999", lw=1, ls="--",
                label="Expected if no subgroup differs")
        ax.scatter(expected, p, s=14, color="#B23A48", zorder=3,
                   label="Observed")
        ax.axhline(0.05, color="#4477AA", lw=0.8, alpha=0.6)
        ax.text(0.985, 0.058, "p = 0.05", ha="right", va="bottom",
                fontsize=6.5, color="#4477AA")

        n_sig = int((p < 0.05).sum())
        ax.set_xlabel("Expected p-value under the null")
        ax.set_ylabel("Observed p-value")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.legend(frameon=False, loc="lower left",
                  bbox_to_anchor=(0, 1.02), ncol=2, fontsize=7.5)
        save(fig, "fig9_heterogeneity_qq")

  wrote fig9_heterogeneity_qq


## Figure 10 — The cohort gradient

Group-time effects by cohort at two horizons. A gradient confined to the longer
horizon, where later cohorts reach the edge of the data, would indicate
incomplete coverage of recent years rather than a difference between cohorts.

In [11]:
ca = load(R13_COHORT)
if ca is None:
    print("cohort ATTs missing; figure 10 skipped")
else:
    hs = [h for h in (3, HORIZON) if h in set(ca.event_time)]
    if not hs:
        print("no cohort effects at the reporting horizons; figure 10 skipped")
    else:
        fig, axes = plt.subplots(1, len(hs), figsize=(4.3 * len(hs), 3.6),
                                 sharey=True, squeeze=False)
        axes = axes[0]
        for ax, h in zip(axes, hs):
            sub = ca[ca.event_time == h]
            for arm in ARMS:
                s = sub[sub.arm == arm].sort_values("cohort")
                if s.empty:
                    continue
                ax.plot(s.cohort, s.att, color=COLOUR[arm], lw=1.5,
                        marker=MARKER[arm], ms=5, label=LABEL[arm])
            ax.axhline(0, color="#333333", lw=0.7)
            ax.set_title(f"at +{h}", loc="left", fontsize=9, pad=8)
            ax.set_xlabel("Retraction cohort")
            ax.xaxis.set_major_locator(MultipleLocator(1))

        axes[0].set_ylabel("Effect on the probability\nof publishing")
        axes[0].legend(frameon=False, loc="lower right", fontsize=7.5)
        save(fig, "fig10_cohort_gradient")

  wrote fig10_cohort_gradient
